# 🕸️ Banco de Dados de Grafos (NoSQL)

> **Missão:** Mostrar o poder de navegar por conexões físicas (Arestas) em vez de fazer JOINs pesados em tabelas cruzadas.
> **Arquivo:** A nossa rede social sintética será salva no arquivo `conexoes_sociais.csv` (representando Origem $\rightarrow$ Destino).

Em bancos relacionais, descobrir os "amigos dos amigos" exige queries com múltiplos `JOINs`, o que consome muita memória. Em um banco de Grafos, o banco simplesmente "pula" de um nó para o outro seguindo a linha física da conexão. No entanto, pedir para um banco de grafos calcular uma métrica global para a rede inteira é o seu calcanhar de Aquiles.

In [1]:
import networkx as nx
import pandas as pd
import os

file_path = 'conexoes_sociais.csv'

if not os.path.exists(file_path):
    print("⏳ Gerando 'conexoes_sociais.csv' no disco (isso pode levar alguns segundos)...")
    
    # Criando uma rede social simulada (Modelo Barabási-Albert cria "influenciadores")
    # 50.000 usuários, cada novo usuário se conecta a 5 pessoas existentes
    G = nx.barabasi_albert_graph(50000, 5, seed=42)
    
    # Extraindo as arestas (relacionamentos)
    arestas = list(G.edges())
    df_arestas = pd.DataFrame(arestas, columns=['origem', 'destino'])
    df_arestas['relacao'] = 'SEGUE'
    
    # Salvando no disco
    df_arestas.to_csv(file_path, index=False)
        
    print(f"✅ Arquivo salvo! Foram geradas {len(df_arestas):,} conexões na rede.")
else:
    print("✅ Arquivo 'conexoes_sociais.csv' já existe. Pronto para as consultas!")

⏳ Gerando 'conexoes_sociais.csv' no disco (isso pode levar alguns segundos)...
✅ Arquivo salvo! Foram geradas 249,975 conexões na rede.


In [2]:
import time
import pandas as pd
import networkx as nx
import ipywidgets as widgets
from ipywidgets import interact

# 1. Carregando o grafo do disco para a memória
df_arestas = pd.read_csv('conexoes_sociais.csv')
G = nx.from_pandas_edgelist(df_arestas, source='origem', target='destino', edge_attr='relacao', create_using=nx.DiGraph())

# 2. Função de Visualização
def exibir_resultado(query: str, titulo: str, explicacao: str, tempo_ms: float, nos_visitados: int, resultado: str):
    print(f"\n{'=' * 80}")
    print("⚙️  PAINEL DE TESTES NOSQL (GRAFOS)")
    print(f"{'=' * 80}")

    print("\n💻 QUERY (CYPHER SIMULADO)")
    print(f"{'-' * 80}")
    print(query.strip())

    print("\n📊 RESUMO")
    print(f"{'-' * 80}")
    print(f"Tempo de execução : {tempo_ms:.2f} ms")
    print(f"Nós visitados     : {nos_visitados:,}".replace(",", "."))
    print(f"Resultado         : {titulo}")

    print("\n📝 O QUE ACONTECEU?")
    print(f"{'-' * 80}")
    print(explicacao)

    print("\n📋 RESULTADO DA CONSULTA")
    print(f"{'-' * 80}")
    print(resultado)

    print(f"\n{'=' * 80}")


# 3. Lógica de Execução
def testar_grafos(tipo_query: str):
    inicio = time.time()

    if tipo_query == "Eficiente (Travessia Local de Conexões)":
        query = """
                MATCH (u:Usuario {id: 500})-[:SEGUE]->(amigo)-[:SEGUE]->(amigo_do_amigo)
                RETURN count(amigo_do_amigo)
                """
        # Simulando a travessia: Amigos dos amigos do usuário 500
        amigos = list(G.successors(500))
        amigos_dos_amigos = set()
        for amigo in amigos:
            amigos_dos_amigos.update(G.successors(amigo))
            
        resultado = f"O usuário 500 possui {len(amigos)} amigos diretos e {len(amigos_dos_amigos)} conexões de 2º grau."
        
        titulo = "🚀 Index-Free Adjacency (Saltos de Ponteiro)"
        explicacao = (
            "O banco ignorou as outras 49.999 pessoas da rede. Ele simplesmente pegou o Nó 500 "
            "e andou pelos 'fios' físicos que conectam ele aos seus amigos. É uma busca "
            "focada e cirúrgica, cujo tempo não aumenta mesmo que a rede tenha 1 bilhão de pessoas."
        )
        nos_visitados = len(amigos) + len(amigos_dos_amigos)
        
    else:
        query = """
                MATCH (n:Usuario)
                CALL algo.pageRank(n) YIELD score
                RETURN max(score)
                """
        # Simulando uma query analítica global (Ex: Calcular a importância de todos os nós)
        # O PageRank força a travessia de TODOS os nós e TODAS as arestas da rede repetidas vezes.
        pagerank_scores = nx.pagerank(G, max_iter=10)
        no_mais_famoso = max(pagerank_scores, key=pagerank_scores.get)
        
        resultado = f"O nó mais influente da rede é o usuário {no_mais_famoso}."
        
        titulo = "🚨 Cálculo Global em Grafo"
        explicacao = (
            "Grafos são excelentes para buscas locais, mas sofrem com queries globais.\n"
            "Pedir para o banco olhar para todos os nós simultaneamente e calcular suas "
            "influências força a leitura da rede inteira múltiplas vezes, destruindo a performance."
        )
        nos_visitados = len(G.nodes)

    fim = time.time()
    tempo_ms = (fim - inicio) * 1000

    exibir_resultado(query, titulo, explicacao, tempo_ms, nos_visitados, resultado)


# 4. Interface Gráfica
opcoes_cenarios = [
    "Eficiente (Travessia Local de Conexões)",
    "Ineficiente (Cálculo Global na Rede)"
]

interact(
    testar_grafos,
    tipo_query=widgets.RadioButtons(
        options=opcoes_cenarios,
        description="Cenário:",
        layout={'width': 'max-content'}
    )
)

interactive(children=(RadioButtons(description='Cenário:', layout=Layout(width='max-content'), options=('Efici…

<function __main__.testar_grafos(tipo_query: str)>